# Taiho-CFD 后处理

当前 `saveMeshData` 每个 MPI rank 只写 owned columns，不包含 ghost columns。因此 rank 文件必须直接沿 x 方向拼接，不能再按旧程序裁掉两列。坐标保持求解器约定的 `y=0` 到 `y=Ly` 方向。

In [ ]:
from pathlib import Path
import sys

TOOLS = Path.cwd() / 'tools'
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))

from postprocess import (
    load_and_combine_data, save_combined_data, save_tecplot, save_vtk,
    _save_plots,
)

DATA_DIR = Path('result')
OUTPUT_DIR = DATA_DIR / 'postprocess'
RANKS = None  # 例如 4；None 表示自动发现连续 rank 文件

fields = load_and_combine_data(DATA_DIR, RANKS)
save_combined_data(fields, OUTPUT_DIR)
save_vtk(fields, OUTPUT_DIR)
save_tecplot(fields, OUTPUT_DIR)
_save_plots(fields, OUTPUT_DIR, show=True)
print('combined shape:', fields['u'].shape)

命令行等价用法：
```bash
python3 tools/postprocess.py --data-dir result --ranks 4
python3 tools/postprocess.py --data-dir result --ranks 4 --no-plots
```

输出包括 `*_combined.dat`、保留非均匀 cell-center 坐标的 `result.vtk`、Tecplot `result.plt`，以及三张 PNG 图。默认包含物理边界单元；如需裁边，应在独立分析脚本中明确处理。